# 🔑 Solutions: Lesson 02 · Layers, Weights & Activation Functions

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 2.1: Count the parameters  &nbsp; `⭐`

Write `count_params(sizes)` for a fully connected network whose layer sizes are in the list `sizes` (e.g. `[784, 128, 64, 10]`). Include biases.


In [2]:
def count_params(sizes):
    return sum(a * b + b for a, b in zip(sizes[:-1], sizes[1:]))

In [3]:
# ▶ self-check
dl.check("784-128-64-10 has 109,386 params", count_params([784,128,64,10]) == 109386)
dl.check("2-2-1 XOR net has 9 params", count_params([2,2,1]) == 9)
dl.check("single layer 5→3 has 18", count_params([5,3]) == 18)

✅ 784-128-64-10 has 109,386 params
✅ 2-2-1 XOR net has 9 params
✅ single layer 5→3 has 18


---
### 🧩 Problem 2.2: Stable softmax  &nbsp; `⭐⭐`

Implement `softmax(z)` that works on a **batch**: `z` has shape `(m, k)`; each *row* must become a probability distribution. It must not overflow for logits like `1000`.


In [4]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

In [5]:
# ▶ self-check
z = np.array([[2., 1., .1], [1000., 1001., 1002.]])
p = softmax(z)
dl.check("no NaN / inf", p is not None and np.isfinite(p).all())
dl.check("each row sums to 1", p is not None and np.allclose(p.sum(1), 1))
dl.check("first row correct", p is not None and np.allclose(p[0], [0.659, 0.242, 0.099], atol=1e-3))
dl.check("shift invariance", p is not None and np.allclose(softmax(z + 5), p))

✅ no NaN / inf
✅ each row sums to 1
✅ first row correct
✅ shift invariance


**Why this works.** Subtracting a constant from every logit in a row leaves softmax unchanged (the factor cancels), but prevents `exp` from overflowing.

---
### 🧩 Problem 2.3: Forward pass of a whole network  &nbsp; `⭐⭐`

Write `forward(X, params)` where `params` is a list of `(W, b)` tuples. Apply **ReLU after every layer except the last**
(the last layer stays linear; those are the logits). Return the logits.


In [6]:
def forward(X, params):
    A = X
    for i, (W, b) in enumerate(params):
        A = A @ W + b
        if i < len(params) - 1:
            A = np.maximum(0, A)
    return A

In [7]:
# ▶ self-check
rng = np.random.default_rng(0)
params = [(rng.normal(size=(4, 6)), rng.normal(size=6)), (rng.normal(size=(6, 3)), rng.normal(size=3))]
X = rng.normal(size=(7, 4))
out = forward(X, params)
expected = np.maximum(0, X @ params[0][0] + params[0][1]) @ params[1][0] + params[1][1]
dl.check("output shape (7, 3)", out is not None and out.shape == (7, 3))
dl.check("matches manual computation", out is not None and np.allclose(out, expected))

✅ output shape (7, 3)
✅ matches manual computation


---
### 🧩 Problem 2.4: Hand-build XOR with ReLU  &nbsp; `⭐⭐⭐`

Find weights for a **2 → 2 → 1 network with ReLU hidden units and a linear output** that computes XOR exactly on the four binary inputs
(outputs 0, 1, 1, 0). Fill `W1 (2×2)`, `b1 (2,)`, `W2 (2×1)`, `b2 (1,)`.


In [8]:
W1 = np.array([[1., 1.], [1., 1.]]); b1 = np.array([0., -1.])
W2 = np.array([[1.], [-2.]]); b2 = np.array([0.])

In [9]:
# ▶ self-check
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
try:
    out = (np.maximum(0, X @ W1 + b1) @ W2 + b2).ravel()
    dl.check("XOR outputs are [0,1,1,0]", np.allclose(out, [0, 1, 1, 0]))
except Exception as e:
    dl.check("XOR outputs are [0,1,1,0]", False, f"({type(e).__name__}) fill in all four arrays")

✅ XOR outputs are [0,1,1,0]


**Why this works.** Two hinges at different positions create a 'bump': increasing, then *decreasing*: exactly what XOR needs and a straight line can't give.

---
### 🧩 Problem 2.5: Activation derivatives  &nbsp; `⭐⭐`

Implement the **derivative** functions `d_sigmoid(z)`, `d_tanh(z)`, `d_relu(z)` (we'll need them for backpropagation).
Facts: $\sigma'(z)=\sigma(z)(1-\sigma(z))$, $\tanh'(z)=1-\tanh^2(z)$, ReLU′ = 1 if z>0 else 0.
Verify numerically in the self-check.


In [10]:
def d_sigmoid(z):
    s = 1 / (1 + np.exp(-z)); return s * (1 - s)
def d_tanh(z):
    return 1 - np.tanh(z) ** 2
def d_relu(z):
    return (z > 0).astype(float)

In [11]:
# ▶ self-check
z = np.linspace(-3, 3, 13) + 0.013; h = 1e-6
sig = lambda z: 1/(1+np.exp(-z))
num = lambda f: (f(z + h) - f(z - h)) / (2 * h)
try:
    dl.check("sigmoid′ matches numerical", np.allclose(d_sigmoid(z), num(sig), atol=1e-5))
    dl.check("tanh′ matches numerical", np.allclose(d_tanh(z), num(np.tanh), atol=1e-5))
    dl.check("ReLU′ matches numerical", np.allclose(d_relu(z), num(lambda t: np.maximum(0, t)), atol=1e-5))
    dl.check("max of sigmoid′ is 0.25", np.isclose(d_sigmoid(np.array([0.0]))[0], 0.25))
except Exception:
    dl.check("implement all three functions", False)

✅ sigmoid′ matches numerical
✅ tanh′ matches numerical
✅ ReLU′ matches numerical
✅ max of sigmoid′ is 0.25
